# WC_MERCURY_BADGEEVENT_F ETL - ODI to Databricks Migration
### Badge Events Fact
**Source Table:** `workspace.PRXBI_TS.WC_MERCURY_BADGEEVENT_TS`
**Target Table:** `workspace.PRXBI_DW.WC_MERCURY_BADGEEVENT_F`
**Detection Strategy:** NONE (always upsert, no column comparison)

#### Migration Notes
- Oracle `PRXBI_DW_SEP` mapped to `workspace.PRXBI_DW`
- Oracle `PRXBI_TS_SEP` mapped to `workspace.PRXBI_TS`
- `SYSTIMESTAMP` replaced with `CURRENT_TIMESTAMP()`
- `NVL()` replaced with `COALESCE()`
- Oracle `/*+ append */` hints and `NOLOGGING` removed
- Oracle indexes removed (Delta handles via Z-ORDER)
- `DBMS_STATS` replaced with `OPTIMIZE` + `ZORDER`
- Oracle sequences (`SEQ.NEXTVAL`) replaced with `BIGINT GENERATED ALWAYS AS IDENTITY`
- Oracle `||` concatenation replaced with `CONCAT_WS('~', ...)`
- Separate UPDATE + INSERT replaced with MERGE INTO
- `SYS_GUID()` / `rowid` / `sysdate` not needed (error tables replaced with PK check)
- All tables use Delta format

In [ ]:
%sql
-- Step 1: Create Widgets for ETL Parameters
CREATE WIDGET TEXT ETL_JOB_TYPE DEFAULT 'EOD';
CREATE WIDGET TEXT DATASOURCE_NUM_ID DEFAULT '380';
CREATE WIDGET TEXT ETL_PROC_WID DEFAULT '1';

## Step 2: Get ETL Parameters
Read `wc_etl_parameters` to obtain extract time windows and current ROW_WID.

In [ ]:
%sql
-- Step 2a: Create temp view for last extract time
CREATE OR REPLACE TEMP VIEW v_etl_last_extract_time AS
SELECT
  COALESCE(
    MAX(CASE WHEN PARAM_NAME = 'LAST_EXTRACT_TIME' THEN CAST(PARAM_VALUE AS TIMESTAMP) END),
    CAST('1900-01-01 00:00:00' AS TIMESTAMP)
  ) AS last_extract_time
FROM workspace.PRXBI_DW.wc_etl_parameters
WHERE DATASOURCE_NUM_ID = ${DATASOURCE_NUM_ID};

In [ ]:
%sql
-- Step 2b: Create temp view for current extract time
CREATE OR REPLACE TEMP VIEW v_etl_current_extract_time AS
SELECT
  COALESCE(
    MAX(CASE WHEN PARAM_NAME = 'CURRENT_EXTRACT_TIME' THEN CAST(PARAM_VALUE AS TIMESTAMP) END),
    CURRENT_TIMESTAMP()
  ) AS current_extract_time
FROM workspace.PRXBI_DW.wc_etl_parameters
WHERE DATASOURCE_NUM_ID = ${DATASOURCE_NUM_ID};

In [ ]:
%sql
-- Step 2c: Create temp view for max ROW_WID
CREATE OR REPLACE TEMP VIEW v_etl_row_wid AS
SELECT COALESCE(MAX(ROW_WID), 0) AS max_row_wid
FROM workspace.PRXBI_DW.WC_MERCURY_BADGEEVENT_F;

In [ ]:
%sql
-- Step 2d: Display all parameters for validation
SELECT 'last_extract_time' AS param, CAST(last_extract_time AS STRING) AS value FROM v_etl_last_extract_time
UNION ALL
SELECT 'current_extract_time' AS param, CAST(current_extract_time AS STRING) AS value FROM v_etl_current_extract_time
UNION ALL
SELECT 'max_row_wid' AS param, CAST(max_row_wid AS STRING) AS value FROM v_etl_row_wid
UNION ALL
SELECT 'ETL_JOB_TYPE' AS param, '${ETL_JOB_TYPE}' AS value
UNION ALL
SELECT 'ETL_PROC_WID' AS param, '${ETL_PROC_WID}' AS value;

## Step 3: Create C$ Staging Table
Drop and recreate the staging table `c_0badgeevent_stg` to hold deduplicated source records from `WC_MERCURY_BADGEEVENT_TS`.
Maps to ODI C$ table `C$_0A10DA20FTVLUG38H7LVMMI5D4D`.

In [ ]:
%sql
-- Step 3a: Drop staging table if exists
DROP TABLE IF EXISTS workspace.PRXBI_DW.c_0badgeevent_stg;

In [ ]:
%sql
-- Step 3b: Create staging table with 8 source columns
CREATE TABLE workspace.PRXBI_DW.c_0badgeevent_stg (
  EVENTEDITIONGBSCODE   STRING,
  EVENTTYPE             STRING,
  BADGEID               STRING,
  SOURCE                STRING,
  PRODUCTCODE           STRING,
  CUSTOMERTYPE          STRING,
  EVENTDATE             STRING,
  CREATEDDATE           STRING
) USING DELTA;

## Step 4: Extract & Deduplicate Source Data
Insert into staging using complex deduplication:
1. Inner subquery: GROUP BY 6 key columns with MAX(INT_INSERT_DATE), filtered by incremental time window
2. INNER JOIN back to source on all 6 key columns + INT_INSERT_DATE with same time filter
3. GROUP BY all 8 columns (adding EVENTDATE, CREATEDDATE)
4. Apply ROW_NUMBER() OVER(PARTITION BY 6 key columns ORDER BY EVENTDATE DESC)
5. Keep only RN=1

In [ ]:
%sql
-- Step 4a: Insert deduplicated source data into staging
INSERT INTO workspace.PRXBI_DW.c_0badgeevent_stg
SELECT
  EVENTEDITIONGBSCODE,
  EVENTTYPE,
  BADGEID,
  SOURCE,
  PRODUCTCODE,
  CUSTOMERTYPE,
  EVENTDATE,
  CREATEDDATE
FROM (
  SELECT
    grp.EVENTEDITIONGBSCODE,
    grp.EVENTTYPE,
    grp.BADGEID,
    grp.SOURCE,
    grp.PRODUCTCODE,
    grp.CUSTOMERTYPE,
    grp.EVENTDATE,
    grp.CREATEDDATE,
    ROW_NUMBER() OVER (
      PARTITION BY grp.EVENTEDITIONGBSCODE, grp.EVENTTYPE, grp.BADGEID,
                   grp.SOURCE, grp.PRODUCTCODE, grp.CUSTOMERTYPE
      ORDER BY grp.EVENTDATE DESC
    ) AS RN
  FROM (
    SELECT
      TS.EVENTEDITIONGBSCODE,
      TS.EVENTTYPE,
      TS.BADGEID,
      TS.SOURCE,
      TS.PRODUCTCODE,
      TS.CUSTOMERTYPE,
      TS.EVENTDATE,
      TS.CREATEDDATE
    FROM workspace.PRXBI_TS.WC_MERCURY_BADGEEVENT_TS TS
    INNER JOIN (
      SELECT
        EVENTEDITIONGBSCODE,
        EVENTTYPE,
        BADGEID,
        SOURCE,
        PRODUCTCODE,
        CUSTOMERTYPE,
        MAX(INT_INSERT_DATE) AS MAX_INT_INSERT_DATE
      FROM workspace.PRXBI_TS.WC_MERCURY_BADGEEVENT_TS
      WHERE INT_INSERT_DATE > (SELECT last_extract_time FROM v_etl_last_extract_time)
        AND INT_INSERT_DATE <= (SELECT current_extract_time FROM v_etl_current_extract_time)
      GROUP BY EVENTEDITIONGBSCODE, EVENTTYPE, BADGEID, SOURCE, PRODUCTCODE, CUSTOMERTYPE
    ) MX
      ON TS.EVENTEDITIONGBSCODE = MX.EVENTEDITIONGBSCODE
     AND TS.EVENTTYPE = MX.EVENTTYPE
     AND TS.BADGEID = MX.BADGEID
     AND TS.SOURCE = MX.SOURCE
     AND TS.PRODUCTCODE = MX.PRODUCTCODE
     AND TS.CUSTOMERTYPE = MX.CUSTOMERTYPE
     AND TS.INT_INSERT_DATE = MX.MAX_INT_INSERT_DATE
    WHERE TS.INT_INSERT_DATE > (SELECT last_extract_time FROM v_etl_last_extract_time)
      AND TS.INT_INSERT_DATE <= (SELECT current_extract_time FROM v_etl_current_extract_time)
    GROUP BY
      TS.EVENTEDITIONGBSCODE,
      TS.EVENTTYPE,
      TS.BADGEID,
      TS.SOURCE,
      TS.PRODUCTCODE,
      TS.CUSTOMERTYPE,
      TS.EVENTDATE,
      TS.CREATEDDATE
  ) grp
) dedup
WHERE RN = 1;

In [ ]:
%sql
-- Step 4b: Validate staging record count
SELECT COUNT(*) AS staging_record_count FROM workspace.PRXBI_DW.c_0badgeevent_stg;

## Step 5: Create I$ Flow Table
Drop and recreate the flow table `i_wc_mercury_badgeevent_f_flow` which holds enriched records
with dimension lookups and an `IND_UPDATE` flag ('I' for insert, 'U' for update).
Maps to ODI I$ table `I$_WAS3QUD00L5IANIT76RN9FTQD96`.

In [ ]:
%sql
-- Step 5a: Drop flow table if exists
DROP TABLE IF EXISTS workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow;

In [ ]:
%sql
-- Step 5b: Create flow table with all columns
CREATE TABLE workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow (
  ROW_WID               BIGINT,
  INTEGRATION_ID        STRING,
  ETL_PROC_WID          BIGINT,
  EVENTEDITIONGBSCODE   STRING,
  EVENTTYPE             STRING,
  BADGEID               STRING,
  SOURCE                STRING,
  PRODUCTCODE           STRING,
  CUSTOMERTYPE          STRING,
  EVENTDATE             STRING,
  CREATEDDATE           STRING,
  BADGE_WID             BIGINT,
  EVENT_EDITION_WID     BIGINT,
  OBU_WID               BIGINT,
  PRODUCT_WID           BIGINT,
  W_UPDATE_DT           TIMESTAMP,
  W_INSERT_DT           TIMESTAMP,
  EVENT_WID             BIGINT,
  IND_UPDATE            STRING
) USING DELTA;

## Step 6: Populate Flow Table with Dimension Lookups
Insert into the flow table with:
- `INTEGRATION_ID` built via `CONCAT_WS('~', ...)` on 6 key columns
- LEFT OUTER JOIN to `WC_EVENT_ED_D` on `RPAD(EVENT_ALPHA_CODE,5,'-') || EVENT_EDITION_CODE`
- LEFT OUTER JOIN to `WC_BADGE_DETAILS_D` on `BADGE_ID`
- LEFT OUTER JOIN to `WC_BADGE_PRODUCT_D` (aggregated by SKU, MAX ROW_WID) on `PRODUCTCODE`
- LEFT OUTER JOIN to `WC_EVENT_D` on `EVENT_INTEGRATION_ID`
- All `NVL(x,0)` replaced with `COALESCE(x,0)`
- `IND_UPDATE = 'I'` (detection strategy NONE)

In [ ]:
%sql
-- Step 6a: Insert into flow table with dimension lookups
INSERT INTO workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow
SELECT
  0                                                       AS ROW_WID,
  CONCAT_WS('~',
    S.EVENTEDITIONGBSCODE,
    S.EVENTTYPE,
    S.BADGEID,
    S.SOURCE,
    S.PRODUCTCODE,
    S.CUSTOMERTYPE
  )                                                       AS INTEGRATION_ID,
  CAST(${ETL_PROC_WID} AS BIGINT)                         AS ETL_PROC_WID,
  S.EVENTEDITIONGBSCODE,
  S.EVENTTYPE,
  S.BADGEID,
  S.SOURCE,
  S.PRODUCTCODE,
  S.CUSTOMERTYPE,
  S.EVENTDATE,
  S.CREATEDDATE,
  COALESCE(BD.ROW_WID, 0)                                 AS BADGE_WID,
  COALESCE(EED.ROW_WID, 0)                                AS EVENT_EDITION_WID,
  COALESCE(EED.OBU_WID, 0)                                AS OBU_WID,
  COALESCE(BP.ROW_WID_1, 0)                               AS PRODUCT_WID,
  CURRENT_TIMESTAMP()                                     AS W_UPDATE_DT,
  CURRENT_TIMESTAMP()                                     AS W_INSERT_DT,
  COALESCE(ED.ROW_WID, 0)                                 AS EVENT_WID,
  'I'                                                     AS IND_UPDATE
FROM workspace.PRXBI_DW.c_0badgeevent_stg S
LEFT OUTER JOIN workspace.PRXBI_DW.WC_EVENT_ED_D EED
  ON S.EVENTEDITIONGBSCODE = CONCAT(RPAD(EED.EVENT_ALPHA_CODE, 5, '-'), EED.EVENT_EDITION_CODE)
LEFT OUTER JOIN workspace.PRXBI_DW.WC_BADGE_DETAILS_D BD
  ON S.BADGEID = BD.BADGE_ID
LEFT OUTER JOIN (
  SELECT SKU AS SKU_1, MAX(ROW_WID) AS ROW_WID_1
  FROM workspace.PRXBI_DW.WC_BADGE_PRODUCT_D
  GROUP BY SKU
) BP
  ON S.PRODUCTCODE = BP.SKU_1
LEFT OUTER JOIN workspace.PRXBI_DW.WC_EVENT_D ED
  ON EED.EVENT_INTEGRATION_ID = ED.INTEGRATION_ID;

In [ ]:
%sql
-- Step 6b: Validate flow table record count
SELECT COUNT(*) AS flow_record_count FROM workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow;

## Step 7: PK Validation (Duplicate Check)
Check for duplicate `INTEGRATION_ID` values in the flow table.
If duplicates exist, deduplicate by keeping the record with the latest `EVENTDATE` per `INTEGRATION_ID`.
This replaces the ODI E$ error table and `SNP_CHECK_TAB` logic.

In [ ]:
%sql
-- Step 7a: Check for duplicate INTEGRATION_IDs
SELECT INTEGRATION_ID, COUNT(*) AS dup_count
FROM workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow
GROUP BY INTEGRATION_ID
HAVING COUNT(*) > 1;

In [ ]:
%sql
-- Step 7b: Deduplicate flow table if needed
-- Create a deduped copy, delete from flow, insert back deduped, then drop temp
CREATE TABLE workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_deduped USING DELTA AS
SELECT *
FROM (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY INTEGRATION_ID
      ORDER BY EVENTDATE DESC
    ) AS rn
  FROM workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow
) ranked
WHERE rn = 1;

DELETE FROM workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow;

INSERT INTO workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow
SELECT
  ROW_WID, INTEGRATION_ID, ETL_PROC_WID,
  EVENTEDITIONGBSCODE, EVENTTYPE, BADGEID, SOURCE, PRODUCTCODE, CUSTOMERTYPE,
  EVENTDATE, CREATEDDATE,
  BADGE_WID, EVENT_EDITION_WID, OBU_WID, PRODUCT_WID,
  W_UPDATE_DT, W_INSERT_DT, EVENT_WID, IND_UPDATE
FROM workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_deduped;

DROP TABLE IF EXISTS workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_deduped;

## Step 8: Mark Updates
Set `IND_UPDATE = 'U'` for records in the flow table where a matching `INTEGRATION_ID`
already exists in the target table. These records will be applied as updates rather than inserts.

In [ ]:
%sql
-- Step 8a: Flag records for update where they already exist in target
UPDATE workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow F
SET IND_UPDATE = 'U'
WHERE EXISTS (
  SELECT 1
  FROM workspace.PRXBI_DW.WC_MERCURY_BADGEEVENT_F T
  WHERE T.INTEGRATION_ID = F.INTEGRATION_ID
);

In [ ]:
%sql
-- Step 8b: Validate IND_UPDATE breakdown
SELECT IND_UPDATE, COUNT(*) AS record_count
FROM workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow
GROUP BY IND_UPDATE
ORDER BY IND_UPDATE;

## Step 9: MERGE to Target
MERGE replaces the separate UPDATE + INSERT from ODI.
- **WHEN MATCHED AND IND_UPDATE = 'U':** Update all fact columns + `ETL_PROC_WID` + `W_UPDATE_DT = CURRENT_TIMESTAMP()`.
- **WHEN NOT MATCHED AND IND_UPDATE = 'I':** Insert new records (ROW_WID generated by IDENTITY column) + `ETL_PROC_WID` + timestamps.

In [ ]:
%sql
-- Step 9: MERGE INTO target from flow table
MERGE INTO workspace.PRXBI_DW.WC_MERCURY_BADGEEVENT_F T
USING workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow S
ON T.INTEGRATION_ID = S.INTEGRATION_ID

WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET
  T.EVENTEDITIONGBSCODE   = S.EVENTEDITIONGBSCODE,
  T.EVENTTYPE             = S.EVENTTYPE,
  T.BADGEID               = S.BADGEID,
  T.SOURCE                = S.SOURCE,
  T.PRODUCTCODE           = S.PRODUCTCODE,
  T.CUSTOMERTYPE          = S.CUSTOMERTYPE,
  T.EVENTDATE             = S.EVENTDATE,
  T.CREATEDDATE           = S.CREATEDDATE,
  T.BADGE_WID             = S.BADGE_WID,
  T.EVENT_EDITION_WID     = S.EVENT_EDITION_WID,
  T.OBU_WID               = S.OBU_WID,
  T.PRODUCT_WID           = S.PRODUCT_WID,
  T.EVENT_WID             = S.EVENT_WID,
  T.ETL_PROC_WID          = S.ETL_PROC_WID,
  T.W_UPDATE_DT           = CURRENT_TIMESTAMP()

WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT (
  INTEGRATION_ID,
  ETL_PROC_WID,
  EVENTEDITIONGBSCODE,
  EVENTTYPE,
  BADGEID,
  SOURCE,
  PRODUCTCODE,
  CUSTOMERTYPE,
  EVENTDATE,
  CREATEDDATE,
  BADGE_WID,
  EVENT_EDITION_WID,
  OBU_WID,
  PRODUCT_WID,
  EVENT_WID,
  W_INSERT_DT,
  W_UPDATE_DT
) VALUES (
  S.INTEGRATION_ID,
  S.ETL_PROC_WID,
  S.EVENTEDITIONGBSCODE,
  S.EVENTTYPE,
  S.BADGEID,
  S.SOURCE,
  S.PRODUCTCODE,
  S.CUSTOMERTYPE,
  S.EVENTDATE,
  S.CREATEDDATE,
  S.BADGE_WID,
  S.EVENT_EDITION_WID,
  S.OBU_WID,
  S.PRODUCT_WID,
  S.EVENT_WID,
  CURRENT_TIMESTAMP(),
  CURRENT_TIMESTAMP()
);

## Step 10: Optimize
Replace Oracle `DBMS_STATS.GATHER_TABLE_STATS` with Delta `OPTIMIZE` and `ZORDER BY` for query performance.

In [ ]:
%sql
-- Step 10: Optimize target table with ZORDER for frequently filtered columns
OPTIMIZE workspace.PRXBI_DW.WC_MERCURY_BADGEEVENT_F
ZORDER BY (INTEGRATION_ID, EVENT_EDITION_WID, BADGE_WID);

## Step 11: Cleanup
Drop the staging (C$) and flow (I$) tables used during the ETL process.

In [ ]:
%sql
-- Step 11: Drop staging, flow, and deduped tables
DROP TABLE IF EXISTS workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow;
DROP TABLE IF EXISTS workspace.PRXBI_DW.c_0badgeevent_stg;
DROP TABLE IF EXISTS workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_deduped;

## Step 12: Validation
Verify the target table record count and inspect sample records to confirm the load completed successfully.

In [ ]:
%sql
-- Step 12a: Final validation
SELECT
  COUNT(*)                        AS total_records,
  COUNT(DISTINCT INTEGRATION_ID)  AS distinct_events,
  MIN(W_INSERT_DT)                AS earliest_insert,
  MAX(W_UPDATE_DT)                AS latest_update,
  SUM(CASE WHEN W_UPDATE_DT > W_INSERT_DT THEN 1 ELSE 0 END) AS updated_records
FROM workspace.PRXBI_DW.WC_MERCURY_BADGEEVENT_F;

In [ ]:
%sql
-- Step 12b: Sample records
SELECT *
FROM workspace.PRXBI_DW.WC_MERCURY_BADGEEVENT_F
ORDER BY W_UPDATE_DT DESC
LIMIT 20;

## Conversion Notes (ODI to Databricks)

| ODI / Oracle Construct | Databricks / Spark SQL Equivalent |
|---|---|
| `PRXBI_DW_SEP` schema | `workspace.PRXBI_DW` |
| `PRXBI_TS_SEP` schema | `workspace.PRXBI_TS` |
| `SYSTIMESTAMP` | `CURRENT_TIMESTAMP()` |
| `NVL(col, val)` | `COALESCE(col, val)` |
| `/*+ append */` hint | Removed (Delta handles append natively) |
| `NOLOGGING` | Removed (Delta manages transaction logs) |
| Oracle indexes | Removed (Delta uses Z-ORDER for data skipping) |
| `DBMS_STATS.GATHER_TABLE_STATS` | `OPTIMIZE` + `ZORDER BY` |
| Oracle sequence for `ROW_WID` | `BIGINT GENERATED ALWAYS AS IDENTITY` on target table |
| `#GLOBAL.v_ETL_JOB_TYPE` | Widget `${ETL_JOB_TYPE}` |
| `#SALES_AND_MARKETING.ETLProcWID` | Widget `${ETL_PROC_WID}` |
| `TO_TIMESTAMP('#GLOBAL...','YYYY-MM-DD HH24:MI:SS.FF')` | Subselect from temp view |
| Oracle `\|\|` concatenation | `CONCAT_WS('~', ...)` in Spark SQL |
| `SYS_GUID()` / `rowid` / `sysdate` | Not needed (error tables replaced with PK check) |
| Separate UPDATE + INSERT | `MERGE INTO ... WHEN MATCHED ... WHEN NOT MATCHED` |
| `VARCHAR2` | `STRING` |
| `NUMBER(x,y)` | `DECIMAL(x,y)` or `BIGINT` |
| `TIMESTAMP(7)` | `TIMESTAMP` |
| C$ table `C$_0A10DA20FTVLUG38H7LVMMI5D4D` | `workspace.PRXBI_DW.c_0badgeevent_stg` (Delta) |
| I$ table `I$_WAS3QUD00L5IANIT76RN9FTQD96` | `workspace.PRXBI_DW.i_wc_mercury_badgeevent_f_flow` (Delta) |
| E$ error table / `SNP_CHECK_TAB` | Duplicate INTEGRATION_ID check + ROW_NUMBER dedup |
| Detection Strategy: NONE | Always upsert - no column comparison needed |
| `RPAD()` in Oracle | `RPAD()` works in Spark SQL too |
| Inner GROUP BY + JOIN back dedup | Preserved as nested subquery with ROW_NUMBER |
| 4 dimension LEFT OUTER JOINs | Preserved: WC_EVENT_ED_D, WC_BADGE_DETAILS_D, WC_BADGE_PRODUCT_D (aggregated), WC_EVENT_D |